# CS5489 - Machine Learning
# Lecture 11a - Large Language Models
### Dept. of Computer Science, City University of Hong Kong

# Outline
1. Tokenization
2. Language Models (LMs)
3. Large Language Models (LLMs)
4. LLMs: Prolems and Mitigations
5. LLMs: Application and Future

In [1]:
# setup
%matplotlib inline
import matplotlib_inline   # setup output image format
matplotlib_inline.backend_inline.set_matplotlib_formats('retina')
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 100  # display larger images
import matplotlib

import numpy as np
import torch
import transformers
import nltk  # for English tokenizer
import jieba  # for Chinese tokenizer

# nltk.download('punkt_tab')

/Users/zzs/miniconda3/envs/cs5489/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Tokenization
- How to represent natural language (text) inputs?
    - We need to first split the input into a sequence of **tokens**; this process is called Tokenization.
- Why not simply split by spaces?
    - There are many langauges that do not split words/tokens by spaces, such as Mandarin, Thai, etc.
    - Even in English, there is a growing number of new words (such as "LLM", "ChatGPT") and compunding words (such as "GPT-based"), which can explode the vocabulary.
- In modern NLP/LLM models, we split the input into a sequence of **subwords (tokens)**.
    - This is the unit that most APIs/Models count the LLM usage: https://help.openai.com/en/articles/4936856-what-are-tokens-and-how-to-count-them 

## Tokenization Method 1: Rule-based
- Split by Spaces + Special Handling of Punctuations
- This used to be the default (and a reasonably good) method for English text processing (before the wide adoptation of subword tokenization)
- NLTK has good support for this: https://www.nltk.org/api/nltk.tokenize.word_tokenize.html

In [2]:
sentence = "I'm studying machine learning and LLM-based methods in City University of Hong Kong."
words = nltk.tokenize.word_tokenize(sentence)
print(words)

['I', "'m", 'studying', 'machine', 'learning', 'and', 'LLM-based', 'methods', 'in', 'City', 'University', 'of', 'Hong', 'Kong', '.']


## Tokenization Method 2: Dictionary/Model-based
- For languages that there are no easy rules to split the sentence, we may treat it as a specific NLP task and use a dictionary or even learn a model to do this.
- There is a traditional NLP task for Chinese called Chinese Word Segmentation: http://sighan.cs.uchicago.edu/bakeoff2005/
- "jieba" is a widely used segmentation library for Chinese: https://github.com/fxsjy/jieba

In [3]:
sentence = "我在香港城市大学学习机器学习以及大模型。"
words0 = nltk.tokenize.word_tokenize(sentence)  # this does not work!
print(words0)
words = jieba.cut(sentence)
print(list(words))

Building prefix dict from the default dictionary ...
Loading model from cache /var/folders/5h/9z8cfrvj02324js7fw73_cb00000gn/T/jieba.cache


['我在香港城市大学学习机器学习以及大模型。']


Loading model cost 0.265 seconds.
Prefix dict has been built successfully.


['我', '在', '香港城市大学', '学习', '机器', '学习', '以及', '大', '模型', '。']


## Tokenization Method 3: Statistics-based
- Currently, (almost all) LLMs use statistics-based tokenization approaches for tokenization.
- Byte-pair encoding (BPE) is one of the mostly widely utilized approach: https://en.wikipedia.org/wiki/Byte-pair_encoding
- Sennrich et al. (2016) introduce the idea for neural machine translation and it is later widely utilized in NLP: https://aclanthology.org/P16-1162.pdf
- The idea is very simple: repeating the process of merging most frequently appearing nearby subtokens.

In [4]:
tokenizer = transformers.AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
sentence_en = "I'm studying machine learning and LLM-based methods in City University of Hong Kong."
sentence_zh = "我在香港城市大学学习机器学习以及大模型。"
tokens_en = tokenizer.tokenize(sentence_en)
tokens_zh = tokenizer.tokenize(sentence_zh)  # the subword representation is not explicit in the original script 
print([tokenizer.convert_tokens_to_string([z]) for z in tokens_en])
print([tokenizer.convert_tokens_to_string([z]) for z in tokens_zh])

['I', "'m", ' studying', ' machine', ' learning', ' and', ' L', 'LM', '-based', ' methods', ' in', ' City', ' University', ' of', ' Hong', ' Kong', '.']
['我在', '香港', '城市', '大学', '学习', '机器', '学习', '以及', '大', '模型', '。']


# Langauge Models (LMs)
- After tokenization, we have our inputs: a sequence of tokens $T=[t_0, t_1, ..., t_{N-1}]$, the task of LM is to build a probablistic model $p(T)$ for the whole sequence.
- First, we need some good representations of the tokens so that they can be easily processed by the machine.
- The bag-of-words representation is not suitable here since we care about the full sequence and each token's position (it is an ordered sequence, not a bag).
- Decomposition: $p(T) = p(t_0)*p(t_1|t_0)*p(t_2|t_0,t_1)*...*p(t_{N-1}|t_0,t_1,...,t_{N-2})$
- Therefore, the main work is to model the **Next Token Prediction**: $p(t_i|t_0,....,t_{i-1})$

## LM Method 1: N-gram LM
- LM can be as simple as counting, but we need some **independence assumptions** here.
- The N-gram LM assume that $p(t_i|t_0,...,t_{i-1}) \approx p(t_i|t_{i-n},...,t_{i-1})$.
- Unigram: $p(t_i)$, Bi-gram: $p(t_i|t_{i-1})$, Tri-gram: $p(t_i|t_{i-2},t_{i-1})$, ...
- For LM evaluation, we usually calculate perplexity (PPL): https://en.wikipedia.org/wiki/Perplexity
    - Take 2 as the base: $PPL = 2^{- \frac{1}{N} \sum_i \log_2 p(t_i)}$

In [5]:
# let's build some simple n-gram LM
from collections import Counter, defaultdict
corpus = [["I", "am", "a", "student"], ["I", "study", "LLM"], ["LLM", "is", "cool"]]

# count everything
unigram, bigram = Counter(), defaultdict(Counter)
for seq in corpus:
    prev = "<BOS>"  # special beginning token
    for tok in seq + ["<EOS>"]:  # add special ending token
        unigram[tok] += 1
        bigram[prev][tok] += 1
        prev = tok

# normalization
prob_unigram = {k: v/sum(unigram.values()) for k,v in unigram.items()}
prob_bigram = {p: {k: v/sum(d.values()) for k,v in d.items()} for p, d in bigram.items()}

print(prob_unigram)
print(prob_bigram)

{'I': 0.15384615384615385, 'am': 0.07692307692307693, 'a': 0.07692307692307693, 'student': 0.07692307692307693, '<EOS>': 0.23076923076923078, 'study': 0.07692307692307693, 'LLM': 0.15384615384615385, 'is': 0.07692307692307693, 'cool': 0.07692307692307693}
{'<BOS>': {'I': 0.6666666666666666, 'LLM': 0.3333333333333333}, 'I': {'am': 0.5, 'study': 0.5}, 'am': {'a': 1.0}, 'a': {'student': 1.0}, 'student': {'<EOS>': 1.0}, 'study': {'LLM': 1.0}, 'LLM': {'<EOS>': 0.5, 'is': 0.5}, 'is': {'cool': 1.0}, 'cool': {'<EOS>': 1.0}}


In [6]:
# calculate the prob of a seq
def get_logprobs(s):
    logprobs = []
    prev = "<BOS>"  # special beginning token
    for tok in s + ["<EOS>"]:
        prob = prob_bigram.get(prev, {}).get(tok, 0.)
        logprobs.append(np.log2(prob).item())
        prev = tok
    return logprobs

for seq in (["I", "am", "a", "student"], ["student", "is", "cool"]):
    _logprobs = get_logprobs(seq)
    _nll = - np.mean(_logprobs)
    _ppl = np.exp2(_nll)
    print(f"Sequence={seq}, Logprobs={_logprobs}, NLL={_nll}, PPL={_ppl}")

Sequence=['I', 'am', 'a', 'student'], Logprobs=[-0.5849625007211563, -1.0, 0.0, 0.0, 0.0], NLL=0.31699250014423125, PPL=1.2457309396155174
Sequence=['student', 'is', 'cool'], Logprobs=[-inf, -inf, 0.0, 0.0], NLL=inf, PPL=inf


/var/folders/5h/9z8cfrvj02324js7fw73_cb00000gn/T/ipykernel_23235/1567174493.py:7: RuntimeWarning: divide by zero encountered in log2
  logprobs.append(np.log2(prob).item())


### What are the main problems of n-gram LMs?
- Sparsity problem: The estimations of low-frequence words or n-grams are poor because of lack of data, for OOV words and phrases, the probabilities will be zero and we need smoothing: https://aclanthology.org/P96-1041.pdf
- Representation problem: we cannot well represent the semantic meaning of the tokens, and we need (neural) distributed representations
- Limited-context problem: we have to make strong assumptions that words only depend on nearby tokens, and this assumption is not true in many times; we need stronger models that can take more context into considerations (such as self-attention based models)

## LM Method 2: (Early) Neural LM
- The idea is to adopt the distributed representations for each token to allow better modeling of the semantics.
- The core innovation is the **word embedding**, which utilizes a vector to represent each word/token.
- Two ways to understand it: 1) a dictionary from a word index to a word vector, 2) sparse matrix multiplication with one-hot input

In [7]:
# create some random init word embeddings
dictionary = {k: i for i, k in enumerate(prob_unigram.keys())}
print(dictionary)
word_embeddings = np.random.random([len(dictionary), 100])  # [vocab_size, embed_dim]

# map to a list of embeddings
seq = ["I", "am", "a", "student", "<EOS>"]
arr_emb = np.stack([word_embeddings[dictionary[t]] for t in seq], 0)
print(arr_emb.shape)

# matrix multiplication
arr_input = np.zeros([len(seq), len(dictionary)])  # [seq_length, vocab_size]
for i, s in enumerate(seq):
    arr_input[i][dictionary[s]] = 1.  # one-hot
arr_emb2 = np.matmul(arr_input, word_embeddings)
print(arr_emb2.shape)

{'I': 0, 'am': 1, 'a': 2, 'student': 3, '<EOS>': 4, 'study': 5, 'LLM': 6, 'is': 7, 'cool': 8}
(5, 100)
(5, 100)


After the embedding layer, we map the original text input to a hidden-vector styled "neural" representations, and we can then use our favourite neural models to deal with them. An MLP can be used together with the local window constraint to make a neural N-gram LM. The output is a softmax over all the tokens in the vocab for the next token prediction.

One of the earlist neural LM: https://www.jmlr.org/papers/volume3/bengio03a/bengio03a.pdf

<center><img src="imgs/fig_llm_nlm.png" width=800></center>

There is one more problem, how to obtain the word embeddings. Of course, you can random initilize them and train the full the LM together with other parameters in the NN, which is basically what we have done previously. Another way is that you can pre-train word vectors with simpler objectives:
- Word2Vec: https://arxiv.org/pdf/1301.3781, https://arxiv.org/pdf/1310.4546
- Glove: https://nlp.stanford.edu/projects/glove/

Let's take a look at the famous word2vec (special neural LM without hidden layers) and the interesting properties of the learned word vectors:
- We do not actually need a LM to predict the next token, so we can freely predict tokens based on the surrounding tokens (not just previous ones)
- There are interesting geometric properties of the learned word vectors: "king - man + woman $\approx$ queen"
<center><img src="imgs/fig_llm_wv.png" width=600></center>
<center><img src="imgs/fig_llm_geo.png" width=600></center>

Still the major problem of limited context window cannot be resolved, especially when there are long-range dependencies:
- "The cake that I bought yersterday is"? vs "The book that I bought yersterday is" (choices = ["delicious", "interesting"])
- Here, we need a 6-gram model to differentiate between these two, which is not quite flexible for a static N-gram neural model to deal
- Maybe we need better models?

## LM Method 3: Transformer LM
- To be noted, there have been many other models for better LMs, including RNNs (recurrent neural networds), CNNs, etc.
- But finally, most modern LMs are built upon the Transformer model (attention is all you need): https://papers.neurips.cc/paper/7181-attention-is-all-you-need.pdf
- We have learned the Transformer architecture in previous lectures, let's review its core attention mechanism:

In [8]:
# the simplist version of attention
def attention(query, key, value):
    scale_factor = 1 / (query.size(-1) ** 0.5)
    attn_weight = query @ key.transpose(-2, -1) * scale_factor
    attn_prob = torch.softmax(attn_weight, dim=-1)
    attn_res = attn_prob @ value
    return attn_res

# check with torch's sdpa
t_input = torch.rand([1, 2, 5, 10])  # [batch_size, head_number, seq_length, head_dim]
res1 = attention(t_input, t_input, t_input)
res2 = torch.nn.functional.scaled_dot_product_attention(t_input, t_input, t_input)
print(torch.allclose(res1, res2))

True


- The benefit of self-attention is that it does not need to assume a fixed-size of window of inputs, and it can take any number of inputs and compute a weighted sum of the value vectors for them. In this way, we do not need any independence assumption if using self-attention for LM (of course you can still add some assumptions by masking out long-range tokens).
- There is one more thing to deal with for LM: causal mask, the prediction of the current token CANNOT depend on the future tokens. This can be achieved by adding mask to the attention calculation.

In [9]:
# attention with causal mask
def attention(query, key, value, is_causal=False):
    L, S = query.size(-2), key.size(-2)
    scale_factor = 1 / (query.size(-1) ** 0.5)
    attn_weight = query @ key.transpose(-2, -1) * scale_factor
    if is_causal:
        attn_bias = torch.zeros(L, S, dtype=query.dtype, device=query.device)
        temp_mask = torch.ones(L, S, dtype=torch.bool).tril(diagonal=0)
        attn_bias.masked_fill_(temp_mask.logical_not(), float("-inf"))
        attn_weight = attn_weight + attn_bias
    attn_prob = torch.softmax(attn_weight, dim=-1)
    attn_res = attn_prob @ value
    return attn_res

# check with torch's sdpa
t_input = torch.rand([1, 2, 5, 10])  # [batch_size, head_number, seq_length, head_dim]
res1 = attention(t_input, t_input, t_input, is_causal=True)
res2 = torch.nn.functional.scaled_dot_product_attention(t_input, t_input, t_input, is_causal=True)
print(torch.allclose(res1, res2))

True


The other componets of Transformer-based LMs are the same as other neural models, such as the input embeddings, output softmax, the internal MLPs, etc. Another very good property of Transformer is that it is easy to parallesize and can scale, which ultimately leads to LARGE scaled trained LLMs.

# Large Language Models (LLMs)
- As the name suggests, they are just LMs that are large! Extraordinarily large! (Nothing too magical ...)
- (But of course, there have been many updates on the Transformer architecture: such as acitvation functions, RMSNorm, Rotray position encoding, grouped-query attention, etc)
- There is the **scaling law**: https://arxiv.org/pdf/2001.08361
- They are still LMs (though they are super large), then why they become the game changers?
<center><img src="imgs/fig_llm_sl.png" width=800></center>

## Turning Point 0: ELmo and BERT (2018) w/ Large-scale Pre-training
- Deep bi-directional (LSTM-based) LM pre-training: https://arxiv.org/pdf/1802.05365
- Deep bi-directional masked LM pre-training with Transformer: https://arxiv.org/pdf/1810.04805
- Generative Pre-trained Transformer 1 (GPT-1): https://en.wikipedia.org/wiki/GPT-1
- The "pre-training then fine-tuning paradigm":

<center><img src="imgs/fig_llm_bert.png" width=800></center>

### Turning Point 1: GPT-3 (2020) w/ In-Context Learning
- GPT-3: Language Models are Few-Shot Learners (https://arxiv.org/pdf/2005.14165)
- In-Context Learning w/ Few-shot prompting means that you do NOT need to tune the model's parameters, just provide them with some similar examples in the input context, and they will solve the problems in a similar way. (The mechanism and knowledge for this is already learned in the LM pre-training stage.)
- See this survey for more details (if you are interested): https://arxiv.org/pdf/2301.00234
<center><img src="imgs/fig_llm_icl.png" width=600></center>

- Emergent Abilities of Large Language Models: https://arxiv.org/pdf/2206.07682
<center><img src="imgs/fig_llm_emergent.png" width=800></center>

### Turning Point 2: InstructGPT (2022) w/ RLHF
- Training language models to follow instructions with human feedback: https://arxiv.org/pdf/2203.02155
- A plain LM is less useful; but an LM that can response to human's queries and correspondingly answering is very useful!
- How to obtain such LMs? (We do not have good evaluation metrics for these open-ended problems) -> Use a trained reward model and align things to human preference.
- Three stages of training: 1) SFT, 2) Reward Model, 3) RLHF
<center><img src="imgs/fig_llm_rlhf.png" width=800></center>

#### SFT vs RL
- In SFT (supervised fine-tuning), we are still tuning the LM with the cross-entropy loss, similar to pre-training. We have some fixed dataset with pairs of (prompt, answer), and then tune the models to maximize the predictions of the answering tokens.
- In RL (reinforcement learning), our target is to maximize a specific reward that the LM finally obtain after generating an answer.

### LMs can be trained to follow instructions!
- This means that we can basically turn many of our problems in such query-answer format and directly ask LMs to solve them!
- The remaining part will be just pormpt engineering??
- This finally leads to ChatGPT (2022) and GPT4 (2023) (also including another important aspect of code-based training).

### Turning Point 3: OpenAI-o1 (2024) and Deepseek-R1 (2025) w/ Reasoning
- The idea originates from chain-of-thought (CoT): https://arxiv.org/pdf/2201.11903
- We do not just need LLMs to act as chatbots to chat with us, we need them to solve more complex tasks, which requires "thinking".
- A series of intermediate reasoning steps can significantly improve the ability of LLMs to perform complex reasoning.
<center><img src="imgs/fig_llm_cot.png" width=800></center>

- The main problem is how to obtain such intermediate thinking rationales with high quality?
- RLVR (RL with Verifiable Rewards) is enough: https://arxiv.org/pdf/2501.12948
- Recent LLMs usually have the "think" mode, where will allow the model to perform deep thinking before answering the question.

In [10]:
def my_generate(_messages, _model, _tokenizer, _thinking):
    text = _tokenizer.apply_chat_template(
        _messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=_thinking # Switches between thinking and non-thinking modes. Default is True.
    )
    model_inputs = _tokenizer([text], return_tensors="pt").to(_model.device)
    # conduct text completion
    generated_ids = _model.generate(
        **model_inputs,
        max_new_tokens=512,  # you can set to a larger value to see more outputs
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
    output_str = _tokenizer.decode(output_ids, skip_special_tokens=True).strip("\n")
    return output_str

# let's try a small model
model_name = "Qwen/Qwen3-0.6B"
model = transformers.AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto")
prompt = "Give me a short introduction to deepseek R1."
messages = [{"role": "user", "content": prompt}]

for thinking in [True, False]:
    output_str = my_generate(messages, model, tokenizer, thinking)
    print(f"Think={thinking}, reply={output_str}\n")

`torch_dtype` is deprecated! Use `dtype` instead!


Think=True, reply=<think>
Okay, the user wants a short introduction to DeepSeek R1. First, I need to make sure I understand what DeepSeek R1 is. From what I know, DeepSeek is a deep learning model developed by Alibaba Group. They have released R1 in recent years.

I should start by mentioning that DeepSeek R1 is a high-performance model. It's built on Alibaba's large-scale data and AI infrastructure. Then, I can highlight its key features like support for multiple languages, efficient training, and real-time processing. Also, since it's a model, it's used in various applications like language translation, content creation, and other AI-driven tasks.

I need to keep it concise but informative. Avoid technical jargon if possible, but still convey the importance. Make sure the introduction flows well and includes both the model's capabilities and its applications. Double-check that all key points are covered without being too detailed.
</think>

DeepSeek R1 is a high-performance deep lear

#### LLM Training
- Pre-training: training on vast amount of (unlabeled) web data to gain basic knowledge; the LLM can already do simple in-context learning after this stage.
- Post-training: SFT/RLHF/RLVR/... to allow the LLM to align to human preference, being able to follow instructions and solve more complex problems.

#### LLM Inference
- Inference: Simply predict the next token one at a time (usally need sampling rather than greedy search)
- There are interesting LLM-System problems to allow efficient LLM inference serving: check vllm (https://github.com/vllm-project/vllm) or sglang (https://github.com/sgl-project/sglang) to see more related techniques. 

In [11]:
# what happens underlying model.generate
def simple_generate(m, t, t_input, max_token: int):
    # encoding
    assert len(t_input) == 1, "For simplicity, we only handle one instance!"
    _output = m(t_input)
    ret = []
    # sample one token at one time
    for _ in range(max_token):
        _cache = _output.past_key_values
        _logits = _output.logits
        # sampling the next token
        next_id = torch.multinomial(_logits[0, -1].softmax(-1), 1).item()  # can be replaced with your favourite sampling algorithm
        ret.append(next_id)
        if next_id == t.eos_token_id:
            break  # end for EOS
        # modeling
        _output = m(torch.as_tensor([[next_id]]).to(t_input), past_key_values=_cache)
    return ret
    
# --
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False # Switches between thinking and non-thinking modes. Default is True.
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
output_ids = simple_generate(model, tokenizer, model_inputs['input_ids'], max_token=100)
output_str = tokenizer.decode(output_ids)
print(output_str)

DeepSeek R1 is a mobile device app that provides a comprehensive and intuitive text-to-speech service for users to convert text into speech. It's designed for quick and clear communication, making it ideal for daily tasks and personal use.<|im_end|>


# LLMs: Prolems and Mitigations
LLMs still have many limitations and problems, including hallucination, grounding problems, efficiency issues, lack of intepretability, safety issues, bias and ethics issues, ...

### Hallucination & RAG
- LLMs may enerate false, fabricated, or inaccurate information that is presented as fact. This occurs because LLMs are designed to predict the next most plausible word based on their training data, and they can fill in gaps with incorrect information, even when they are unsure of the correct answer.
- In ordinary LLMs, there are NO explicit memory or knowledge modules, all the knowledge is stored inside the parameters.
- How to deal with long-tail facts and new knowledge?

In [13]:
# here we are guiding the LM to produce false information, but there are many cases the LLMs may hallucinate even in reasonable queries.
messages = [{"role": "user", "content": "Give me the detailed process of how China won the 2008 FIFA world cup."}]
output_str = my_generate(messages, model, tokenizer, False)
print(output_str)

China did not win the 2008 FIFA World Cup. This is a factual and historical fact. The 2008 World Cup was won by **Brazil**, not China. The event took place in **Porto Alegre, Brazil**, and was held from **July 12 to July 15, 2008**.


Retrieval-augmentation generation (RAG) is a technology to augment the context of LLMs with related information from external knowledge base. It can be one of the methods to mitigate hallucination problem of LLMs by providing extra information.

In [18]:
related_information = ["FIFA world cup was held in the following years: ..., 2006, 2010, 2014, 2018, 2022\nFIFA world cup winners: ..., 2006 Italy, 2010 Spain, 2014 Germany, 2018 France, 2022 Argentina"]  # we can use retrieval models to do this

messages = [{"role": "user", "content": f"Here is related information: {related_information}\n\nGive me the detailed process of how China won the 2008 FIFA world cup."}]
output_str = my_generate(messages, model, tokenizer, False)
print(output_str)

China did not win the **2008 FIFA World Cup**. The **2006 FIFA World Cup** was won by **Italy**, the **2010 World Cup** by **Spain**, the **2014 World Cup** by **Germany**, the **2018 World Cup** by **France**, and the **2022 World Cup** by **Argentina**.

If you're looking for information about **China's participation or achievements in other World Cups**, let me know!


## Grounding & Agent
- LLMs themselves can only chat, but cannot do anything further, especially when we need grounding in a specific environment.
- But being used as the central control engine and augmented with perception and action modules, it can work as an agent to solve much more real world tasks.
- Some introductions: https://lilianweng.github.io/posts/2023-06-23-agent/

<center><img src="imgs/fig_llm_agent.png" width=800></center>

In [23]:
input_inst = """You are a helpful agent tasked to perform a web-browsing task.

You will reveive a simplified representation of the current web page's observation and you will need to choose the next action.

The valid actions include:
- click [ID]: click a specific link, the [ID] should be the index of the element
- type [ID] [content]: type "content" into the input box of [ID]
- goto [URL]: goto a specific URL
- finish [answer]: finish the task with the [answer]
"""  # the main instruction

input_obs = """Here is the current observation:
[1] [Text] Google's Homepage
[2] [InputBox] Input your search query
[3] [Button] Search
[4] [Link] advertisement
[5] [Link] Wikipedia: 2022 FIFA World Cup, Argentina were crowned the champions after winning the final against the title holder France 4–2 on penalties following a 3–3 draw after extra time. It was ...
[6] [Link] FIFA's homepage: ...
"""  # the current observation

input_hist = """Here is the history steps that you have finished:
Step 1: Go to Google's Homepage
Step 2: Search with the query "2022 FIFA world cup winner"
"""  # history steps

input_prompt = """The target task is to answer the follwing query:
Who is the winner of 2022 FIFA world cup?

Please generate your action according to the action specifications.
"""

messages = [{"role": "user", "content": "\n".join([input_inst, input_obs, input_hist, input_prompt])}]
output_str = my_generate(messages, model, tokenizer, True)
print(output_str)


<think>
Okay, let's see. The user wants to know who the winner of the 2022 FIFA World Cup was. The current web page shows a few links. The first link is "Google's Homepage", which we already visited. Then there's an input box where they can type a query, and a button to search. Then there are two links: one is "advertisement" and another is "Wikipedia: 2022 FIFA World Cup, Argentina were crowned...".

The history steps so far are step 1: Go to Google's Homepage, step 2: Search with query "2022 FIFA world cup winner". Now, the task is to answer the question about the winner.

Looking at the current observation, the link provided is "Wikipedia: 2022 FIFA World Cup, Argentina were crowned...". So the answer should be Argentina. Since the user's query is asking for the winner, and the link points directly to that information, the correct action would be to click on that link. But wait, the valid actions include clicking on the ID, which in the current observation is [ID]. Let me check the 

(*note*, sometimes, with the small 0.56B model, it still cannot handle the task well. Larger models will get better performance.)

## Efficiency
- LLMs are so large and costly, it will be difficult to train and deploy them.
- Many techniques have been investigated: quantization, compression, distillation, efficient attention.
- This is not only an ML algorithm problem, this is also an ML system problem, where we need model-hardware co-designs.
- A good example is flash attention: https://arxiv.org/pdf/2205.14135 https://github.com/Dao-AILab/flash-attention

<center><img src="imgs/fig_llm_flash.png" width=800></center>

## Intepretability and Safety
- NNs are black-boxes, LLMs are large black-boxes; we still do not fully understand their internal mechanisms.
- This is important for the satety of using them; how to make sure they can reliably generate reasonable outputs and do not go out of control?
- Transformer Circuits Thread: https://transformer-circuits.pub/

<center><img src="imgs/fig_llm_mi.png" width=800></center>

## Bias and Ethics Issues
- Bias Issues: Stereotypes, Representational Bias, Algorithmic Discrimination, Toxic and Harmful Content
- Ethics Issues: Privacy Concerns, Intellectual Property and Copyright, Job Displacement, Environmental Impact
- We need to carefully think more ...

# LLMs: Applications and Future
- Applications:
    - Chatbot
    - agent systems for QA, web, GUI, SWE, even for robots etc
    - personal assistant
    - ...
- Main research questions:
    - Is Transformer the "ultimate" model architecture for LMs?
    - Is next-token-prediction the "ultimate" training method for LMs?
    - How to better incorporate more modalities (image, audio, video, etc) together with the LMs?